# ResNet18 Pretrained Transfer Learning

Purpose: collect the ResNet18 pretrained runs in one notebook. This includes the frozen feature extractor baseline, the augmented frozen run, and the last-block fine-tuning runs on the stricter split seeds.

This notebook reads from `reports/model_comparison/model_results_master.csv`; it does not duplicate the training code.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

master_path = ROOT / 'reports' / 'model_comparison' / 'model_results_master.csv'
summary_path = ROOT / 'reports' / 'model_comparison' / 'model_results_master.json'
if not master_path.exists() or not summary_path.exists():
    display(Markdown("**Model results summary not found. Rebuilding from local metric artifacts...**"))
    subprocess.run([sys.executable, '-m', 'src.evaluation.build_model_results_master'], cwd=ROOT, check=True)

master = pd.read_csv(master_path)
summary = json.loads(summary_path.read_text(encoding='utf-8'))

metric_cols = [
    'model_name', 'run_group', 'pretrained', 'seed', 'split',
    'accuracy', 'precision_macro', 'recall_macro', 'f1_macro',
    'best_epoch', 'epochs_trained', 'recommendation', 'notes'
]

loaded_rows = master[master['status'] == 'loaded'].copy()
missing_rows = master[master['status'] != 'loaded'].copy()
if missing_rows.empty:
    display(Markdown(f"**Artifact status:** all `{len(master)}` result rows loaded from local metrics."))
else:
    display(Markdown(
        f"**Artifact status:** `{len(loaded_rows)}` loaded, `{len(missing_rows)}` missing. "
        "Missing rows mean the ignored metric/checkpoint artifacts are not present locally. "
        "Run the reproduction commands in this notebook, then rerun `python -m src.evaluation.build_model_results_master`."
    ))
    display(missing_rows[['model_family', 'run_group', 'seed', 'status', 'metrics_file']].head(20))

def rebuild_master_results():
    """Rebuild reports/model_comparison from whatever metric artifacts exist locally."""
    subprocess.run([sys.executable, '-m', 'src.evaluation.build_model_results_master'], cwd=ROOT, check=True)
    return pd.read_csv(master_path)

**Artifact status:** `6` loaded, `12` missing. Missing rows mean the ignored metric/checkpoint artifacts are not present locally. Run the reproduction commands in this notebook, then rerun `python -m src.evaluation.build_model_results_master`.

,model_family,run_group,seed,status,metrics_file
4,resnet18,resnet18_scratch_20ep_strict,42.0,missing,reports/resnet18_scratch/resnet18_scratch_stri...
5,resnet18,resnet18_scratch_50ep_early_stopped_strict,42.0,missing,reports/resnet18_scratch/resnet18_scratch_50ep...
7,resnet18,resnet18_scratch_20ep_strict,123.0,missing,reports/resnet18_scratch/resnet18_scratch_stri...
8,resnet18,resnet18_scratch_50ep_early_stopped_strict,123.0,missing,reports/resnet18_scratch/resnet18_scratch_50ep...
10,resnet18,resnet18_scratch_20ep_strict,999.0,missing,reports/resnet18_scratch/resnet18_scratch_stri...
11,resnet18,resnet18_scratch_50ep_early_stopped_strict,999.0,missing,reports/resnet18_scratch/resnet18_scratch_50ep...
12,convnextv2_tiny,convnextv2_pretrained_linear_probe,NaN,missing,model/convnextv2_tiny_fcmae_ft_in1k_linear_pro...
13,convnextv2_tiny,convnextv2_pretrained_linear_probe,123.0,missing,model/convnextv2_tiny_fcmae_ft_in1k_linear_pro...
14,convnextv2_tiny,convnextv2_pretrained_finetune_suspect,NaN,missing,model/convnextv2_tiny_fcmae_ft_in1k_finetune/b...
15,convnextv2_tiny,convnextv2_scratch_50ep_early_stopped_strict,42.0,missing,reports/convnextv2_scratch/convnextv2_tiny_scr...


In [2]:
resnet_pretrained = master[(master['model_family'] == 'resnet18') & (master['pretrained'] == True)].copy()
resnet_pretrained[metric_cols].sort_values(['run_group', 'seed'])

,model_name,run_group,pretrained,seed,split,accuracy,precision_macro,recall_macro,f1_macro,best_epoch,epochs_trained,recommendation,notes
2,ResNet18 fine-tuned last block,resnet18_pretrained_finetune_last_block_origin...,True,NaN,original split_manifest.csv,1.000000,1.000000,1.000000,1.000000,1.0,10.0,strong baseline,Original-split fine-tuned transfer-learning run.
3,ResNet18 fine-tuned last block strict split,resnet18_pretrained_finetune_last_block_strict,True,42.0,strict contiguous split seed 42,1.000000,1.000000,1.000000,1.000000,4.0,10.0,headline transfer-learning comparison,Strict split transfer-learning run for seed ro...
6,ResNet18 fine-tuned last block strict split,resnet18_pretrained_finetune_last_block_strict,True,123.0,strict contiguous split seed 123,0.992857,0.992958,0.992857,0.992831,5.0,10.0,headline transfer-learning comparison,Strict split transfer-learning run for seed ro...
9,ResNet18 fine-tuned last block strict split,resnet18_pretrained_finetune_last_block_strict,True,999.0,strict contiguous split seed 999,1.000000,1.000000,1.000000,1.000000,1.0,10.0,headline transfer-learning comparison,Strict split transfer-learning run for seed ro...
1,ResNet18 frozen feature extractor augmented,resnet18_pretrained_frozen_augmented,True,NaN,original split_manifest.csv,0.994643,0.994706,0.994643,0.994630,NaN,10.0,augmentation comparison,Frozen-feature follow-up run with training-onl...
0,ResNet18 frozen feature extractor,resnet18_pretrained_frozen_no_aug,True,NaN,original split_manifest.csv,0.994643,0.994681,0.994643,0.994643,NaN,10.0,baseline,First transfer-learning baseline without stoch...


In [3]:
resnet_pretrained.groupby('run_group').agg(
    runs=('accuracy', 'count'),
    mean_accuracy=('accuracy', 'mean'),
    min_accuracy=('accuracy', 'min'),
    max_accuracy=('accuracy', 'max'),
    mean_f1=('f1_macro', 'mean'),
).reset_index()

,run_group,runs,mean_accuracy,min_accuracy,max_accuracy,mean_f1
0,resnet18_pretrained_finetune_last_block_origin...,1,1.000000,1.000000,1.000000,1.000000
1,resnet18_pretrained_finetune_last_block_strict,3,0.997619,0.992857,1.000000,0.997610
2,resnet18_pretrained_frozen_augmented,1,0.994643,0.994643,0.994643,0.994630
3,resnet18_pretrained_frozen_no_aug,1,0.994643,0.994643,0.994643,0.994643


## Reproduction Commands

The main strict-split transfer-learning runs are produced with:

```powershell
python -m src.data.create_strict_split_manifests --seeds 42 123 999
python -m src.training.resnet.finetune_last_block --manifest data/splits/strict_split_manifest_seed42.csv --seed 42 --artifact-prefix resnet18_finetune_last_block_strict_seed42
python -m src.training.resnet.finetune_last_block --manifest data/splits/strict_split_manifest_seed123.csv --seed 123 --artifact-prefix resnet18_finetune_last_block_strict_seed123
python -m src.training.resnet.finetune_last_block --manifest data/splits/strict_split_manifest_seed999.csv --seed 999 --artifact-prefix resnet18_finetune_last_block_strict_seed999
```